# ODIR-5K Siamese Swin Transformer Binary Classification (Phase 1)

Notebook này triển khai huấn luyện và so sánh hiệu năng các mô hình Siamese nhị phân (Bình thường vs Bệnh lý) ghép cặp hai mắt đáy mắt y sinh.
- **Mô hình chính:** Siamese Network trích xuất đặc trưng hai mắt đồng bộ.
- **Tiền xử lý:** ROI Crop + Ben Graham Color Normalization + CLAHE.
- **Tăng cường:** Trộn MixUp và CutMix đồng bộ hai mắt y khoa.

In [ ]:
# -- CELL 1: Cài đặt các thư viện cần thiết --
!pip install timm albumentations pyyaml scikit-learn -q
print('✅ Đã cài đặt xong các thư viện')

In [ ]:
# -- CELL 2: Kiểm tra cấu hình GPU --
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('⚠️ Cảnh báo: Không phát hiện GPU!')

In [ ]:
# -- CELL 3: Cấu hình thư mục làm việc (Auto-Detect) --
import os, sys
WORK_CODE = '/kaggle/working/code'
os.makedirs(WORK_CODE, exist_ok=True)

# Tìm kiếm đường dẫn input code
CODE_INPUT = None
for candidate in [
    '/kaggle/input/odir5k-code',
    '/kaggle/input/datasets/ngodinhdatcpp/odir5k-code',
]:
    if os.path.isdir(candidate):
        CODE_INPUT = candidate
        break

if CODE_INPUT:
    print(f'✅ Tìm thấy mã nguồn tại: {CODE_INPUT}')
else:
    raise FileNotFoundError('❌ Không tìm thấy dataset odir5k-code!')

In [ ]:
# -- CELL 4: Đồng bộ hóa mã nguồn vào thư mục làm việc --
import shutil, zipfile

for name in ['src', 'configs', 'splits_clean']:
    dest = f'{WORK_CODE}/{name}'
    if os.path.exists(dest):
        shutil.rmtree(dest)
    src_dir = f'{CODE_INPUT}/{name}'
    src_zip = f'{CODE_INPUT}/{name}.zip'
    if os.path.isdir(src_dir):
        shutil.copytree(src_dir, dest)
    elif os.path.exists(src_zip):
        with zipfile.ZipFile(src_zip) as z:
            z.extractall(WORK_CODE)

for fname in ['train.py', 'evaluate.py']:
    shutil.copy(f'{CODE_INPUT}/{fname}', f'{WORK_CODE}/{fname}')

sys.path.insert(0, WORK_CODE)
CODE_DIR = WORK_CODE
SPLITS_DIR = f'{WORK_CODE}/splits_clean'
print('✅ Mã nguồn đã được chuẩn bị sẵn sàng tại:', CODE_DIR)

In [ ]:
# -- CELL 5: Tìm kiếm ảnh gốc ODIR-5K --
RAW_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'Training Images' in dirs:
        candidate = os.path.join(root, 'Training Images')
        n = len([f for f in os.listdir(candidate) if f.endswith('.jpg')])
        if n > 1000:
            RAW_DIR = candidate
            break

if RAW_DIR:
    print(f'✅ Tìm thấy Training Images tại: {RAW_DIR} ({len(os.listdir(RAW_DIR))} ảnh)')
else:
    raise FileNotFoundError('❌ Không tìm thấy ODIR-5K Training Images!')

ROI_DIR = '/kaggle/working/preprocessed_images'
ENH_DIR = '/kaggle/working/enhanced_images'
RES_DIR = '/kaggle/working/results'
for d in [ROI_DIR, ENH_DIR, RES_DIR]:
    os.makedirs(d, exist_ok=True)

In [ ]:
# -- CELL 6: Chạy tiền xử lý ảnh đáy mắt (ROI Crop & Ben Graham + CLAHE) --
import cv2, numpy as np
from pathlib import Path
from tqdm import tqdm

def crop_roi(img, tol=7):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = gray > tol
    rows, cols = np.where(mask)
    if len(rows) == 0:
        return cv2.resize(img, (512, 512))
    r0, r1, c0, c1 = rows.min(), rows.max(), cols.min(), cols.max()
    return cv2.resize(img[r0:r1+1, c0:c1+1], (512, 512))

def ben_graham(img, sigma_ratio=1/6, scale=128):
    h, w = img.shape[:2]
    sigma = int(max(h, w) * sigma_ratio)
    if sigma % 2 == 0:
        sigma += 1
    local = cv2.GaussianBlur(img.astype(np.float32), (0, 0), sigmaX=sigma)
    return np.clip(img.astype(np.float32) - local + scale, 0, 255).astype(np.uint8)

def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return cv2.cvtColor(cv2.merge([clahe.apply(l), a, b]), cv2.COLOR_LAB2BGR)

imgs = list(Path(RAW_DIR).glob('*.jpg'))
print(f'Bắt đầu tiền xử lý {len(imgs)} ảnh đáy mắt...')

for p in tqdm(imgs, desc='Preprocessing'):
    img = cv2.imread(str(p))
    if img is None: continue
    try:
        # ROI crop trước
        cropped = crop_roi(img)
        cv2.imwrite(str(Path(ROI_DIR)/p.name), cropped)
        
        # Enhance tiếp theo
        enhanced = apply_clahe(ben_graham(cropped))
        cv2.imwrite(str(Path(ENH_DIR)/p.name), enhanced)
    except Exception as e:
        # Fallback basic resize if error
        cv2.imwrite(str(Path(ENH_DIR)/p.name), cv2.resize(img, (512, 512)))

print(f'✅ Hoàn thành tiền xử lý nâng cao. Ảnh lưu tại: {ENH_DIR}')

In [ ]:
# -- CELL 7: Chạy kiểm thử khô (Dry-Run) kiểm tra pipeline --
config_path = f'{CODE_DIR}/configs/exp_6_swin_binary_enhanced_aug.yaml'
cmd = f'PYTHONPATH={CODE_DIR} python3 -u {CODE_DIR}/train.py --config {config_path} --dry-run'
print(f'Executing command: {cmd}')
os.system(cmd)

In [ ]:
# -- CELL 8: Huấn luyện tuần tự 3 thực nghiệm nhị phân song nhãn --
import os, json
binary_configs = [
    'exp_4_swin_binary_raw.yaml',
    'exp_5_swin_binary_enhanced.yaml',
    'exp_6_swin_binary_enhanced_aug.yaml'
]
for cfg_name in binary_configs:
    config_path = f'{CODE_DIR}/configs/{cfg_name}'
    exp_id = cfg_name.replace('.yaml', '')
    print(f'\n============================================================')
    print(f'  BẮT ĐẦU THỰC NGHIỆM: {exp_id}')
    print(f'============================================================')
    cmd = f'PYTHONPATH={CODE_DIR} python3 -u {CODE_DIR}/train.py --config {config_path}'
    print(f'Executing command: {cmd}')
    os.system(cmd)
    
    # Hiển thị kết quả thu được trên Test set
    results_path = f'{RES_DIR}/{exp_id}/test_results.json'
    if os.path.exists(results_path):
        with open(results_path, 'r', encoding='utf-8') as f:
            print(f' Kết quả {exp_id} trên Test Set:')
            print(json.dumps(json.load(f), indent=2))
    else:
        print(f'❌ Không tìm thấy kết quả của thực nghiệm: {exp_id}')

In [ ]:
# -- CELL 9: So sánh kết quả và lập bảng đối chiếu (Ablation Study) --
import os
exps = ['exp_4_swin_binary_raw', 'exp_5_swin_binary_enhanced', 'exp_6_swin_binary_enhanced_aug']
exps_arg = ' '.join([f'{RES_DIR}/{e}' for e in exps])
cmd = f'PYTHONPATH={CODE_DIR} python3 {CODE_DIR}/evaluate.py --exps {exps_arg} --results-dir {RES_DIR}'
print(f'Executing: {cmd}')
os.system(cmd)

# Hiển thị báo cáo
report_path = f'{RES_DIR}/comparison_table.md'
if os.path.exists(report_path):
    with open(report_path, 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print('❌ Không tìm thấy bảng báo cáo so sánh!')

In [ ]:
# -- CELL 10: Vẽ đường cong học tập (Learning Curves) --
import pandas as pd
import matplotlib.pyplot as plt
import os

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71']
exps = ['exp_4_swin_binary_raw', 'exp_5_swin_binary_enhanced', 'exp_6_swin_binary_enhanced_aug']
labels = ['EXP 4: Swin Binary Raw (Ảnh gốc)', 'EXP 5: Swin Binary Enhanced (ROI+BenGraham+CLAHE)', 'EXP 6: Swin Binary Enhanced + Aug (MixUp+CutMix)']

for exp, color, label in zip(exps, colors, labels):
    csv_path = f'{RES_DIR}/{exp}/training_log.csv'
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        axes[0].plot(df['epoch'], df['val_auc_roc'], c=color, label=label, linewidth=2)
        axes[1].plot(df['epoch'], df['val_age_mae'], c=color, label=label, linewidth=2)

axes[0].set(title='Siamese Swin Transformer — Val AUC-ROC', xlabel='Epoch', ylabel='AUC-ROC')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set(title='Siamese Swin Transformer — Val Retinal Age MAE', xlabel='Epoch', ylabel='MAE (years)')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RES_DIR}/swin_learning_curves.png', dpi=150)
plt.show()